# 2 · Feature engineering (computed per timeframe, no cross-TF leakage)

| TF | Purpose | Primary Indicators |
|----|---------|---------------------|
| D1 | Market regime | EMA200, EMA50, ADX, ATR, Ichimoku Kumo (thickness, price-vs-cloud) |
| H4 | Trend confirmation | EMA20, EMA50, ADX, MACD |
| H1 | Setup detection | EMA20, EMA50, RSI, CCI, ATR |
| M15 | Entry & execution | EMA9, EMA20, RSI, CCI, ATR, Bollinger Bands, Session, Time-of-day |

All rolling/EWM ops are causal (no `shift(-1)`, no `center=True`) — each row only uses that bar and earlier ones.

In [8]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from config.settings import DATA_PROCESSED, DATA_FEATURES

SYMBOL = "GBPUSD"
TIMEFRAMES = ["D1", "H4", "H1", "M15"]

clean = {tf: pd.read_parquet(DATA_PROCESSED / f"{SYMBOL}_{tf}.parquet") for tf in TIMEFRAMES}
for tf, df in clean.items():
    print(f"{tf}: {len(df)} rows  ({df['datetime'].iloc[0]} .. {df['datetime'].iloc[-1]})")

D1: 3783 rows  (2012-01-11 00:00:00+00:00 .. 2026-07-10 00:00:00+00:00)
H4: 22680 rows  (2012-01-11 00:00:00+00:00 .. 2026-07-10 00:00:00+00:00)
H1: 90426 rows  (2012-01-11 01:00:00+00:00 .. 2026-07-10 00:00:00+00:00)
M15: 356499 rows  (2012-01-11 01:30:00+00:00 .. 2026-07-10 00:00:00+00:00)


## Shared helpers

In [9]:
def atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    """Average true range (Wilder), causal."""
    prev_close = df["close"].shift(1)
    tr = pd.concat(
        [
            df["high"] - df["low"],
            (df["high"] - prev_close).abs(),
            (df["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    return tr.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()


def ema(series: pd.Series, span: int) -> pd.Series:
    return series.ewm(span=span, adjust=False, min_periods=span).mean()


def rsi(df: pd.DataFrame, period: int = 14) -> pd.Series:
    """Wilder RSI, causal."""
    delta = df["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))


def adx(df: pd.DataFrame, period: int = 14) -> pd.Series:
    """Wilder ADX, causal."""
    up_move = df["high"].diff()
    down_move = -df["low"].diff()

    plus_dm = pd.Series(np.where((up_move > down_move) & (up_move > 0), up_move, 0.0), index=df.index)
    minus_dm = pd.Series(np.where((down_move > up_move) & (down_move > 0), down_move, 0.0), index=df.index)

    tr_atr = atr(df, period)
    plus_di = 100 * plus_dm.ewm(alpha=1 / period, adjust=False, min_periods=period).mean() / tr_atr
    minus_di = 100 * minus_dm.ewm(alpha=1 / period, adjust=False, min_periods=period).mean() / tr_atr

    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
    return dx.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()


def macd(series: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9) -> pd.DataFrame:
    macd_line = ema(series, fast) - ema(series, slow)
    signal_line = macd_line.ewm(span=signal, adjust=False, min_periods=signal).mean()
    return pd.DataFrame({"macd": macd_line, "macd_signal": signal_line, "macd_hist": macd_line - signal_line})


def bollinger_bands(series: pd.Series, period: int = 20, num_std: float = 2.0) -> pd.DataFrame:
    mid = series.rolling(period, min_periods=period).mean()
    std = series.rolling(period, min_periods=period).std()
    upper = mid + num_std * std
    lower = mid - num_std * std
    return pd.DataFrame(
        {
            "bb_mid": mid,
            "bb_upper": upper,
            "bb_lower": lower,
            "bb_width": (upper - lower) / mid,
            "bb_pct_b": (series - lower) / (upper - lower),
        }
    )


def cci(df: pd.DataFrame, period: int = 20) -> pd.Series:
    """Commodity Channel Index, causal."""
    typical_price = (df["high"] + df["low"] + df["close"]) / 3
    sma = typical_price.rolling(period, min_periods=period).mean()
    mean_dev = typical_price.rolling(period, min_periods=period).apply(
        lambda x: np.mean(np.abs(x - x.mean())), raw=True
    )
    return (typical_price - sma) / (0.015 * mean_dev)


def ichimoku_kumo(df: pd.DataFrame, tenkan_p: int = 9, kijun_p: int = 26, senkou_b_p: int = 52) -> pd.DataFrame:
    """Ichimoku Kumo (cloud) — causal, NOT forward-shifted (the traditional 26-period-ahead
    plot would leak future information; these are current-bar values only)."""
    tenkan = (df["high"].rolling(tenkan_p).max() + df["low"].rolling(tenkan_p).min()) / 2
    kijun = (df["high"].rolling(kijun_p).max() + df["low"].rolling(kijun_p).min()) / 2
    senkou_a = (tenkan + kijun) / 2
    senkou_b = (df["high"].rolling(senkou_b_p).max() + df["low"].rolling(senkou_b_p).min()) / 2

    kumo_top = pd.concat([senkou_a, senkou_b], axis=1).max(axis=1)
    kumo_bottom = pd.concat([senkou_a, senkou_b], axis=1).min(axis=1)

    return pd.DataFrame(
        {
            "kumo_thickness": (senkou_a - senkou_b).abs() / df["close"],
            "price_vs_kumo_top": (df["close"] - kumo_top) / df["close"],
        }
    )

## D1 · Market regime — EMA200, EMA50, ADX, ATR

In [10]:
def features_d1(df: pd.DataFrame, adx_period: int = 14, atr_period: int = 14) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["datetime"] = df["datetime"]

    out["ema200"] = ema(df["close"], 200)
    out["ema50"] = ema(df["close"], 50)
    out["ema_regime"] = np.sign(out["ema50"] - out["ema200"])

    out["adx"] = adx(df, adx_period)

    out["atr"] = atr(df, atr_period)
    out["atr_pct"] = out["atr"] / df["close"]

    kumo_df = ichimoku_kumo(df)
    out["kumo_thickness"] = kumo_df["kumo_thickness"]
    out["price_vs_kumo_top"] = kumo_df["price_vs_kumo_top"]

    return out


feat_d1 = features_d1(clean["D1"])
feat_d1.tail()

,datetime,ema200,ema50,ema_regime,adx,atr,atr_pct,kumo_thickness,price_vs_kumo_top
3778,2026-07-06 00:00:00+00:00,1.338479,1.336349,-1.0,23.309882,0.007729,0.005771,0.008047,-0.000373
3779,2026-07-07 00:00:00+00:00,1.338440,1.336281,-1.0,22.262011,0.007581,0.005680,0.007860,-0.003941
3780,2026-07-08 00:00:00+00:00,1.338449,1.336401,-1.0,20.834591,0.007670,0.005726,0.007127,-0.000396
3781,2026-07-09 00:00:00+00:00,1.338478,1.336594,-1.0,19.882115,0.007480,0.005576,0.006482,0.001081
3782,2026-07-10 00:00:00+00:00,1.338502,1.336762,-1.0,18.997673,0.006978,0.005204,0.006136,0.000761


## H4 · Trend confirmation — EMA20, EMA50, ADX, MACD

In [11]:
def features_h4(df: pd.DataFrame, adx_period: int = 14) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["datetime"] = df["datetime"]

    out["ema20"] = ema(df["close"], 20)
    out["ema50"] = ema(df["close"], 50)
    out["ema_trend"] = np.sign(out["ema20"] - out["ema50"])

    out["adx"] = adx(df, adx_period)

    macd_df = macd(df["close"])
    out["macd"] = macd_df["macd"]
    out["macd_signal"] = macd_df["macd_signal"]
    out["macd_hist"] = macd_df["macd_hist"]

    return out


feat_h4 = features_h4(clean["H4"])
feat_h4.tail()

,datetime,ema20,ema50,ema_trend,adx,macd,macd_signal,macd_hist
22675,2026-07-09 08:00:00+00:00,1.337121,1.333524,1.0,26.523894,0.001846,0.001843,0.000002
22676,2026-07-09 12:00:00+00:00,1.337460,1.333804,1.0,26.198301,0.001912,0.001857,0.000055
22677,2026-07-09 16:00:00+00:00,1.337772,1.334076,1.0,26.207114,0.001946,0.001875,0.000071
22678,2026-07-09 20:00:00+00:00,1.338110,1.334360,1.0,25.635229,0.001997,0.001899,0.000098
22679,2026-07-10 00:00:00+00:00,1.338374,1.334616,1.0,25.108404,0.001980,0.001916,0.000065


## H1 · Setup detection — EMA20, EMA50, RSI, ATR

In [12]:
def features_h1(df: pd.DataFrame, rsi_period: int = 14, atr_period: int = 14, cci_period: int = 20) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["datetime"] = df["datetime"]

    out["ema20"] = ema(df["close"], 20)
    out["ema50"] = ema(df["close"], 50)
    ema_diff_sign = np.sign(out["ema20"] - out["ema50"])
    out["ema_cross_up"] = ((ema_diff_sign == 1) & (ema_diff_sign.shift(1) <= 0)).astype(int)
    out["ema_cross_down"] = ((ema_diff_sign == -1) & (ema_diff_sign.shift(1) >= 0)).astype(int)

    out["rsi"] = rsi(df, rsi_period)
    out["cci"] = cci(df, cci_period)

    out["atr"] = atr(df, atr_period)
    out["atr_pct"] = out["atr"] / df["close"]

    return out


feat_h1 = features_h1(clean["H1"])
feat_h1.tail()

,datetime,ema20,ema50,ema_cross_up,ema_cross_down,rsi,cci,atr,atr_pct
90421,2026-07-09 20:00:00+00:00,1.340158,1.338880,0,0,55.365041,37.289892,0.001326,0.000989
90422,2026-07-09 21:00:00+00:00,1.340155,1.338929,0,0,51.680401,-24.925557,0.001351,0.001008
90423,2026-07-09 22:00:00+00:00,1.340252,1.339017,0,0,56.792466,32.998885,0.001334,0.000994
90424,2026-07-09 23:00:00+00:00,1.340354,1.339107,0,0,57.491015,58.585513,0.001257,0.000937
90425,2026-07-10 00:00:00+00:00,1.340405,1.339177,0,0,54.757974,37.033740,0.001200,0.000895


## M15 · Entry & execution — EMA9, EMA20, RSI, ATR, Bollinger Bands

In [13]:
def features_m15(df: pd.DataFrame, rsi_period: int = 14, atr_period: int = 14, bb_period: int = 20, cci_period: int = 20) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["datetime"] = df["datetime"]

    out["ema9"] = ema(df["close"], 9)
    out["ema20"] = ema(df["close"], 20)
    ema_diff_sign = np.sign(out["ema9"] - out["ema20"])
    out["ema_cross_up"] = ((ema_diff_sign == 1) & (ema_diff_sign.shift(1) <= 0)).astype(int)
    out["ema_cross_down"] = ((ema_diff_sign == -1) & (ema_diff_sign.shift(1) >= 0)).astype(int)

    out["rsi"] = rsi(df, rsi_period)
    out["cci"] = cci(df, cci_period)

    out["atr"] = atr(df, atr_period)
    out["atr_pct"] = out["atr"] / df["close"]

    bb_df = bollinger_bands(df["close"], bb_period)
    out["bb_upper"] = bb_df["bb_upper"]
    out["bb_lower"] = bb_df["bb_lower"]
    out["bb_width"] = bb_df["bb_width"]
    out["bb_pct_b"] = bb_df["bb_pct_b"]

    # session / time-of-day — pure calendar math, identical across any data source (no train/live mismatch)
    hour = df["datetime"].dt.hour
    out["session_asian"] = ((hour >= 0) & (hour < 8)).astype(int)
    out["session_london"] = ((hour >= 8) & (hour < 16)).astype(int)
    out["session_ny"] = ((hour >= 13) & (hour < 21)).astype(int)
    out["session_overlap"] = ((hour >= 13) & (hour < 16)).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    return out


feat_m15 = features_m15(clean["M15"])
feat_m15.tail()

,datetime,ema9,ema20,ema_cross_up,ema_cross_down,rsi,cci,atr,atr_pct,bb_upper,bb_lower,bb_width,bb_pct_b,session_asian,session_london,session_ny,session_overlap,hour_sin,hour_cos
356494,2026-07-09 23:00:00+00:00,1.340893,1.340793,0,0,56.123127,61.364415,0.000570,0.000425,1.341904,1.339754,0.001603,0.686597,0,0,0,0,-0.258819,0.965926
356495,2026-07-09 23:15:00+00:00,1.340930,1.340820,0,0,54.055722,48.827204,0.000542,0.000404,1.341893,1.339755,0.001594,0.619789,0,0,0,0,-0.258819,0.965926
356496,2026-07-09 23:30:00+00:00,1.340996,1.340862,0,0,56.143497,63.803936,0.000518,0.000386,1.341878,1.339760,0.001579,0.708240,0,0,0,0,-0.258819,0.965926
356497,2026-07-09 23:45:00+00:00,1.341061,1.340906,0,0,56.847416,78.010949,0.000495,0.000369,1.341842,1.339775,0.001541,0.747571,0,0,0,0,-0.258819,0.965926
356498,2026-07-10 00:00:00+00:00,1.341027,1.340904,0,0,50.581531,42.682927,0.000493,0.000367,1.341789,1.339783,0.001497,0.551820,1,0,0,0,0.000000,1.000000


## Save (each TF's features saved standalone, keyed by its own datetime — no merge)

In [ ]:
feat_by_tf = {"D1": feat_d1, "H4": feat_h4, "H1": feat_h1, "M15": feat_m15}

for tf, df in feat_by_tf.items():
    path = DATA_FEATURES / f"{SYMBOL}_{tf}_features.parquet"
    df.to_parquet(path, index=False)
    print(f"[SAVE] {path.relative_to(DATA_FEATURES.parent.parent)}: {len(df)} rows, {df.shape[1]} cols")